# Phase 4.1 ? OOF diagnostic only (CPU, cache-only)

Attach the official competition data, saved Phase 2 private output, saved fixed Phase 4 output, and Phase 3 delta bundle as before. Internet Off; accelerator CPU. The pinned project wheel is installed offline. All experiment logic runs directly in notebook cells; no external experiment `.py` is written.

This notebook reconstructs Phase 4 OOF, diagnoses every changed consensus answer set and missing gold answer in the cached top 80, then writes **only** `phase41_consensus_diagnostic_report.json`. It does not train or score private candidates, create a submission JSON/ZIP, or recommend submission. The top-5 oracle uses training labels only and is **not** an attainable private estimate. Keep the Phase 4 submission.


In [ ]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'

INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working/legalir-phase41-consensus-private')
RUNTIME = Path('/kaggle/working/legalir-phase41-consensus-runtime')
WORK.mkdir(parents=True, exist_ok=True)
RUNTIME.mkdir(parents=True, exist_ok=True)

def exactly_one(items, label):
    found = list(items)
    if len(found) != 1:
        raise RuntimeError(f'Expected exactly one {label}, found {len(found)}: {found}')
    return found[0]

def read(path):
    return json.loads(path.read_text(encoding='utf-8'))

REPORT_FILE = exactly_one(
    (p for p in INPUT_ROOT.rglob('phase4_report.json')
     if read(p).get('experiment_id') == 'phase4-phase2-legal-bge-protected-blender'
     and (p.parent / 'submission_phase4_private_final.json').is_file()),
    'saved Phase 4 output report',
)
PHASE4_OUTPUT = REPORT_FILE.parent
REPORT = read(REPORT_FILE)
private_matches = [
    p for p in INPUT_ROOT.rglob('private-official.json')
    if p.is_file() and hashlib.sha256(p.read_bytes()).hexdigest() == REPORT['test_sha256']
]
official_matches = [
    p for p in private_matches
    if (p.parent / 'train.json').is_file() and any(p.parent.rglob('context_*.json'))
]
PRIVATE_FILE = exactly_one(official_matches or private_matches, 'official private input matching Phase 4 report')
STATE_FILE = exactly_one(
    (p for p in INPUT_ROOT.rglob('inference_input_state.json')
     if (lambda s: s.get('filename') == 'private-official.json'
         and s.get('sha256') == REPORT['test_sha256']
         and s.get('questions') == REPORT['test_questions']
         and s.get('project_commit') == REPORT['project_commit'])(read(p))
     and (p.parent / 'artifacts_phase2_harrier').is_dir()),
    'matching saved Phase 2 private output',
)
PHASE2_ARTIFACTS = STATE_FILE.parent / 'artifacts_phase2_harrier'
DELTA_MANIFEST = exactly_one(
    (p for p in INPUT_ROOT.rglob('bundle_manifest.json')
     if read(p).get('experiment_id') == 'phase3-rerankers-harrier-retrieval'
     and read(p).get('project_commit') == REPORT['project_commit']),
    'Phase 3 delta bundle with matching project commit',
)
DELTA_ROOT = DELTA_MANIFEST.parents[1]
WHEEL = exactly_one((DELTA_ROOT / 'wheels').glob('uit_legalir-*.whl'), 'offline project wheel')
print('Phase 4 output:', PHASE4_OUTPUT)
print('Phase 2 caches:', PHASE2_ARTIFACTS)
print('Private input:', PRIVATE_FILE)
print('Project wheel:', WHEEL)


In [ ]:
# Offline pinned project wheel; execute all experiment code directly in notebook cells.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
                '--target', str(RUNTIME), str(WHEEL)], check=True)
sys.path.insert(0, str(RUNTIME))
import legalir, sklearn, numpy, yaml
print('CPU dependencies OK', sklearn.__version__)


In [ ]:
from __future__ import annotations

import hashlib
import json
import pickle
import sys
from collections import defaultdict
from pathlib import Path
from typing import Any

import numpy as np
import yaml
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from legalir.fusion import rrf
from legalir.storage import read_json, read_jsonl, write_json
from legalir.text import normalize_question
from legalir.validation import grouped_folds, score_candidates, score_predictions, validate_submission_shape


RERANKERS = ("jina", "vietnamese_reranker", "legal_reranker")
RETRIEVAL_CHANNELS = (
    "bm25",
    "accent_char",
    "vietlegal_harrier",
    "vietnamese_embedding",
    "nemotron",
    "query_memory",
    "query_exact",
)


def fuse_cached(
    retrievals: dict[str, dict[str, Any]],
    weights: dict[str, float],
    rrf_k: int,
    limit: int,
) -> dict[str, dict[str, list[str]]]:
    return {
        qid: {"candidates": rrf(retrieval["channels"], weights, rrf_k, limit)}
        for qid, retrieval in retrievals.items()
    }


def engine_ranks(artifacts: Path, split: str, engine: str, fold: int | None = None) -> dict[str, list[str]]:
    suffix = f"_{fold}" if fold is not None else ""
    separate = artifacts / f"rerank_{split}{suffix}_{engine}.json"
    if separate.is_file():
        return read_json(separate)[engine]
    combined = artifacts / f"rerank_{split}{suffix}.json"
    payload = read_json(combined)
    if engine not in payload:
        raise RuntimeError(f"{engine} is missing from {combined}")
    return payload[engine]


def inner_fold(question: str) -> int:
    # A salt different from grouped_folds is essential: every selected question
    # is already in outer fold zero under the main fold hash.
    key = "phase4-inner-v1:" + normalize_question(question)
    return int(hashlib.sha256(key.encode("utf-8")).hexdigest()[:12], 16) % 5


def reciprocal_rank(rank: int, k: int = 20) -> float:
    return 1.0 / (k + rank)


def rank_features(rank: int, missing_rank: int) -> list[float]:
    clipped = min(rank, missing_rank)
    denominator = max(1, missing_rank - 1)
    return [
        reciprocal_rank(clipped),
        1.0 / clipped,
        1.0 - min(clipped - 1, denominator) / denominator,
        float(clipped <= 5),
        float(clipped <= 10),
        float(clipped <= 20),
        float(clipped <= 50),
        float(clipped < missing_rank),
    ]


def feature_vector(document: str, orders: dict[str, dict[str, int]]) -> list[float]:
    features: list[float] = []
    core_ranks: list[int] = []
    for name in ("first_stage", *RERANKERS):
        rank = orders[name].get(document, 81)
        core_ranks.append(rank)
        features.extend(rank_features(rank, 81))
    retrieval_ranks: list[int] = []
    for name in RETRIEVAL_CHANNELS:
        rank = orders[name].get(document, 151)
        retrieval_ranks.append(rank)
        features.extend(rank_features(rank, 151))

    all_ranks = core_ranks + retrieval_ranks
    features.extend(
        [
            float(sum(rank <= 5 for rank in core_ranks)),
            float(sum(rank <= 10 for rank in core_ranks)),
            float(sum(rank <= 20 for rank in core_ranks)),
            float(sum(rank <= 20 for rank in retrieval_ranks)),
            float(sum(rank < 151 for rank in retrieval_ranks)),
            float(min(all_ranks)),
            float(max(core_ranks)),
            float(np.mean(core_ranks)),
            float(np.std(core_ranks)),
        ]
    )
    phase2_score = (
        0.3 * reciprocal_rank(core_ranks[0])
        + 0.5 * reciprocal_rank(core_ranks[1])
        + 0.5 * reciprocal_rank(core_ranks[2])
    )
    legal_residual = reciprocal_rank(core_ranks[3]) - reciprocal_rank(core_ranks[0])
    features.extend([phase2_score, legal_residual])
    return features


def make_rows(
    questions: list[dict[str, Any]],
    fused: dict[str, dict[str, Any]],
    retrievals: dict[str, dict[str, Any]],
    rerankings: dict[str, dict[str, list[str]]],
    labelled: bool,
) -> tuple[np.ndarray, np.ndarray, list[tuple[str, str]], np.ndarray]:
    rows: list[list[float]] = []
    labels: list[int] = []
    metadata: list[tuple[str, str]] = []
    groups: list[int] = []
    for question in questions:
        qid = question["qid"]
        candidates = fused[qid]["candidates"][:80]
        orders = {"first_stage": {doc: rank for rank, doc in enumerate(candidates, 1)}}
        for name, values in rerankings.items():
            orders[name] = {doc: rank for rank, doc in enumerate(values[qid], 1)}
        for name in RETRIEVAL_CHANNELS:
            orders[name] = {doc: rank for rank, doc in enumerate(retrievals[qid]["channels"][name], 1)}
        truth = set(question.get("answers", []))
        group = inner_fold(question["question"])
        for document in candidates:
            rows.append(feature_vector(document, orders))
            labels.append(int(document in truth) if labelled else 0)
            metadata.append((qid, document))
            groups.append(group)
    return (
        np.asarray(rows, dtype=np.float32),
        np.asarray(labels, dtype=np.int8),
        metadata,
        np.asarray(groups, dtype=np.int8),
    )


def standardize_per_query(values: np.ndarray, metadata: list[tuple[str, str]]) -> np.ndarray:
    result = np.zeros(len(values), dtype=np.float64)
    by_qid: defaultdict[str, list[int]] = defaultdict(list)
    for index, (qid, _) in enumerate(metadata):
        by_qid[qid].append(index)
    for indices in by_qid.values():
        scores = values[indices]
        scale = scores.std()
        result[indices] = (scores - scores.mean()) / (scale if scale > 1e-9 else 1.0)
    return result


def scores_by_question(values: np.ndarray, metadata: list[tuple[str, str]]) -> dict[str, dict[str, float]]:
    output: defaultdict[str, dict[str, float]] = defaultdict(dict)
    for score, (qid, document) in zip(values, metadata, strict=True):
        output[qid][document] = float(score)
    return dict(output)


def protected_predictions(
    blended_scores: np.ndarray,
    metadata: list[tuple[str, str]],
    baseline: dict[str, list[str]],
    margin: float,
) -> tuple[dict[str, list[str]], dict[str, int]]:
    per_query = scores_by_question(blended_scores, metadata)
    output: dict[str, list[str]] = {}
    promoted_questions = 0
    promotions = 0
    for qid, scores in per_query.items():
        selected = list(baseline[qid])
        outsiders = [doc for doc, _ in sorted(scores.items(), key=lambda item: (-item[1], item[0])) if doc not in selected]
        changed = False
        for outsider in outsiders:
            weakest = min(selected, key=lambda doc: (scores[doc], doc))
            if scores[outsider] < scores[weakest] + margin:
                break
            selected[selected.index(weakest)] = outsider
            promotions += 1
            changed = True
        if changed:
            promoted_questions += 1
        # Ordering is irrelevant to Recall, but sorting makes the output deterministic.
        output[qid] = sorted(selected, key=lambda doc: (-scores[doc], doc))
    return output, {"promoted_questions": promoted_questions, "promotions": promotions}


def new_model(c_value: float):
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=c_value,
            class_weight="balanced",
            max_iter=600,
            solver="liblinear",
            random_state=2026,
        ),
    )


def baseline_predictions(
    questions: list[dict[str, Any]],
    fused: dict[str, dict[str, Any]],
    rerankings: dict[str, dict[str, list[str]]],
    final_weights: dict[str, Any],
) -> dict[str, list[str]]:
    return {
        question["qid"]: rrf(
            {
                "first_stage": fused[question["qid"]]["candidates"],
                "jina": rerankings["jina"][question["qid"]],
                "vietnamese_reranker": rerankings["vietnamese_reranker"][question["qid"]],
            },
            final_weights["weights"],
            final_weights["rrf_k"],
            5,
        )
        for question in questions
    }


In [ ]:
"""Diagnose the fixed Phase 4.1 consensus experiment; OOF cache only."""

from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import yaml

from legalir.storage import read_json, read_jsonl, write_json
from legalir.text import clean_text
from legalir.validation import grouped_folds, score_candidates, score_predictions


REPORT_NAME = "phase41_consensus_diagnostic_report.json"
LEGAL_TOP = 5
OTHER_TOP = 10
WEAK_LEGAL_AFTER = 10
SCORE_GAP = 0.25  # Phase 4 scores are standardized within each question.


def required(path: Path) -> Path:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Required cache/output missing or broken symlink: {path}")
    return path


def verify_source(report: dict, private_path: Path, phase4_dir: Path, phase2_dir: Path) -> None:
    content = required(private_path).read_bytes()
    if (report.get("experiment_id") != "phase4-phase2-legal-bge-protected-blender"
            or report.get("test_file") != "private-official.json"
            or report.get("test_sha256") != hashlib.sha256(content).hexdigest()
            or report.get("test_questions") != len(json.loads(content))):
        raise RuntimeError("Phase 4 report does not match the official private questions")
    p4 = read_json(required(phase4_dir / "artifacts_phase4" / "prepare_manifest.json"))
    p2 = read_json(required(phase2_dir / "prepare_manifest.json"))
    for key in ("schema_version", "chunking_fingerprint", "documents", "short_chunks", "long_chunks",
                "train_questions", "public_questions", "train_questions_fingerprint", "public_questions_fingerprint"):
        if p4.get(key) != p2.get(key):
            raise RuntimeError(f"Phase 2/4 prepared data differs: {key}")
    # This diagnostic needs only training caches; private rankings are not inspected.
    for name in ("first_stage_weights.json", "final_weights.json", "retrieval_train.json"):
        required(phase2_dir / name)
    for split, suffix in (("train", "_0"),):
        for engine in ("jina", "vietnamese_reranker"):
            required(phase2_dir / f"rerank_{split}{suffix}_{engine}.json")
        required(phase4_dir / "artifacts_phase4" / f"rerank_{split}{suffix}_legal_reranker.json")
    for name in ("train_questions.jsonl",):
        required(phase4_dir / "artifacts_phase4" / name)
    for name in ("kaggle_rtx_pro_6000_phase4.yaml",):
        required(phase4_dir / name)


def per_inner(predictions: dict[str, list[str]], questions: list[dict]) -> dict[str, dict[str, float]]:
    return {
        str(fold): score_predictions(
            predictions, [q for q in questions if inner_fold(q["question"]) == fold]
        ) for fold in range(5)
    }


def gate(candidate: dict, phase4: dict, tolerance: float = 1e-12) -> tuple[bool, str]:
    if candidate["metrics"]["recall"] <= phase4["metrics"]["recall"] + tolerance:
        return False, "Consensus OOF Recall did not exceed Phase 4"
    for fold in range(5):
        key = str(fold)
        if candidate["per_inner"][key]["recall"] < phase4["per_inner"][key]["recall"] - tolerance:
            return False, f"Consensus OOF Recall declined in inner fold {fold}"
    return True, "OOF Recall improved with no declining inner fold"


def oof_scores(x: np.ndarray, y: np.ndarray, groups: np.ndarray,
               metadata: list[tuple[str, str]]) -> np.ndarray:
    raw = np.empty(len(y), dtype=np.float64)
    for fold in range(5):
        train, valid = groups != fold, groups == fold
        if not valid.any() or len(np.unique(y[train])) != 2:
            raise RuntimeError(f"Missing inner fold or class: {fold}")
        model = new_model(0.1)
        model.fit(x[train], y[train])
        raw[valid] = model.decision_function(x[valid])
    return standardize_per_query(raw, metadata)


def consensus_predictions(phase4: dict[str, list[str]], scores: np.ndarray,
                          metadata: list[tuple[str, str]],
                          ranks: dict[str, dict[str, list[str]]]) -> tuple[dict[str, list[str]], dict[str, int]]:
    """At most one swap per query; never consult labels or private outcomes."""
    per_query = scores_by_question(scores, metadata)
    if set(phase4) != set(per_query):
        raise RuntimeError("Phase 4 QIDs and score QIDs differ")
    result: dict[str, list[str]] = {}
    changed = 0
    eligible = 0
    for qid, selected in phase4.items():
        values = per_query[qid]
        if len(selected) != 5 or len(set(selected)) != 5 or not set(selected) <= set(values):
            raise RuntimeError(f"Invalid Phase 4 top five: {qid}")
        positions = {name: {doc: i for i, doc in enumerate(ranks[name][qid], 1)}
                     for name in RERANKERS}
        if any(set(pos) != set(values) for pos in positions.values()):
            raise RuntimeError(f"Consensus ranking/score mismatch: {qid}")
        weakest = min(selected, key=lambda doc: (values[doc], doc))
        weak_unprotected = (positions["legal_reranker"][weakest] > WEAK_LEGAL_AFTER
                            and positions["jina"][weakest] > OTHER_TOP
                            and positions["vietnamese_reranker"][weakest] > OTHER_TOP)
        outsiders = [doc for doc in values if doc not in selected
                     and positions["legal_reranker"][doc] <= LEGAL_TOP
                     and (positions["jina"][doc] <= OTHER_TOP
                          or positions["vietnamese_reranker"][doc] <= OTHER_TOP)
                     and values[doc] >= values[weakest] - SCORE_GAP]
        if weak_unprotected and outsiders:
            eligible += 1
            outsider = min(outsiders, key=lambda doc: (-values[doc], doc))
            updated = list(selected)
            updated[updated.index(weakest)] = outsider
            result[qid] = updated
            changed += 1
        else:
            result[qid] = list(selected)
    return result, {"eligible_questions": eligible, "changed_answer_sets": changed}


def compare_phase4(predictions: dict[str, list[str]], saved_submission: dict) -> int:
    if set(predictions) != set(saved_submission):
        raise RuntimeError("Saved Phase 4 submission QIDs differ from reconstructed QIDs")
    changed = sum(set(documents) != set(saved_submission[qid]["answer"])
                  for qid, documents in predictions.items())
    return changed


def diagnose_oof(questions: list[dict], phase4: dict[str, list[str]],
                 consensus: dict[str, list[str]], fused: dict[str, dict]) -> dict:
    """Describe the unchanged fixed rule; no parameter search or private labels."""
    qids = {q["qid"] for q in questions}
    if len(qids) != len(questions) or set(phase4) != qids or set(consensus) != qids:
        raise RuntimeError("OOF question/prediction QIDs differ")
    counts = {key: 0 for key in ("improved", "harmed", "unchanged")}
    by_fold = {str(i): {"improved": 0, "harmed": 0, "unchanged": 0,
                        "changed": 0, "recall_delta": 0.0} for i in range(5)}
    swaps = []
    missed_by_question = []
    missed = {"questions_with_misses": 0, "questions_with_miss_in_top80": 0,
              "questions_with_miss_outside_top80": 0, "missed_answers_in_top80": 0,
              "missed_answers_outside_top80": 0, "questions_with_empty_truth": 0}
    candidate_recall_sum = 0.0
    top5_oracle_sum = 0.0
    delta_sum = 0.0
    for question in questions:
        qid = question["qid"]
        truth = set(question["answers"])
        old, new = set(phase4[qid]), set(consensus[qid])
        candidates = fused[qid]["candidates"][:80]
        if (len(phase4[qid]) != 5 or len(consensus[qid]) != 5 or len(old) != 5 or
                len(new) != 5 or len(candidates) != 80 or len(set(candidates)) != 80 or
                not old <= set(candidates) or not new <= set(candidates)):
            raise RuntimeError(f"Invalid OOF top 5/top 80: {qid}")
        if not truth:
            missed["questions_with_empty_truth"] += 1
        else:
            top80_truth = truth.intersection(candidates)
            candidate_recall_sum += len(top80_truth) / len(truth)
            top5_oracle_sum += min(5, len(top80_truth)) / len(truth)
            absent = truth - old
            inside = absent.intersection(candidates)
            outside = absent - inside
            if absent:
                missed["questions_with_misses"] += 1
            missed["questions_with_miss_in_top80"] += bool(inside)
            missed["questions_with_miss_outside_top80"] += bool(outside)
            missed["missed_answers_in_top80"] += len(inside)
            missed["missed_answers_outside_top80"] += len(outside)
            if absent:
                missed_by_question.append({"qid": qid, "inner_fold": inner_fold(question["question"]),
                                           "missed_in_top80": sorted(inside),
                                           "missed_outside_top80": sorted(outside)})
        if old == new:
            continue
        removed, added = old - new, new - old
        if len(removed) != 1 or len(added) != 1:
            raise RuntimeError(f"Consensus must make exactly one swap: {qid}")
        removed_doc, added_doc = next(iter(removed)), next(iter(added))
        hit_delta = int(added_doc in truth) - int(removed_doc in truth)
        recall_delta = hit_delta / len(truth) if truth else 0.0
        outcome = "improved" if recall_delta > 0 else "harmed" if recall_delta < 0 else "unchanged"
        fold = str(inner_fold(question["question"]))
        counts[outcome] += 1
        by_fold[fold][outcome] += 1
        by_fold[fold]["changed"] += 1
        by_fold[fold]["recall_delta"] += recall_delta
        delta_sum += recall_delta
        swaps.append({"qid": qid, "inner_fold": int(fold), "true_answers": sorted(truth),
                      "removed_doc": removed_doc,
                      "added_doc": added_doc, "removed_is_answer": removed_doc in truth,
                      "added_is_answer": added_doc in truth, "outcome": outcome,
                      "recall_delta": recall_delta})
    count = len(questions)
    return {"changed_answer_sets": len(swaps), "outcomes": counts, "per_inner": by_fold,
            "swaps": swaps, "mean_recall_delta": delta_sum / count,
            "missed_phase4": missed, "missed_phase4_by_question": missed_by_question, "candidate_recall_top80": candidate_recall_sum / count,
            "top5_oracle_recall_upper_bound": top5_oracle_sum / count}


def main(phase4_dir: Path, phase2_dir: Path, work: Path, private_path: Path) -> None:
    work.mkdir(parents=True, exist_ok=True)
    # Remove both prior diagnostic and previously generated consensus ZIP/JSON.
    for name in (REPORT_NAME, "phase41_consensus_report.json",
                 "submission_phase41_private_consensus.json",
                 "submission_phase41_private_consensus.zip"):
        stale = work / name
        if stale.exists():
            stale.unlink()
    report = read_json(required(phase4_dir / "phase4_report.json"))
    verify_source(report, private_path, phase4_dir, phase2_dir)
    config = yaml.safe_load(required(phase4_dir / "kaggle_rtx_pro_6000_phase4.yaml").read_text(encoding="utf-8"))
    if config["validation"]["folds"] != 5 or config["retrieval"]["fused_top_k"] < 80:
        raise RuntimeError("Phase 4 fold count or candidate depth differs")
    p4_artifacts = phase4_dir / "artifacts_phase4"
    retrieval_train = read_json(phase2_dir / "retrieval_train.json")
    first_stage = read_json(phase2_dir / "first_stage_weights.json")
    weights = read_json(phase2_dir / "final_weights.json")
    fused_train = fuse_cached(retrieval_train, first_stage["weights"], first_stage["rrf_k"],
                              config["retrieval"]["fused_top_k"])
    train_questions = list(read_jsonl(p4_artifacts / "train_questions.jsonl"))
    fold_questions = [q for q in train_questions if grouped_folds([q], 5)[q["qid"]] == 0]
    train_ranks = {name: engine_ranks(phase2_dir if name != "legal_reranker" else p4_artifacts,
                                     "train", name, 0) for name in RERANKERS}
    questions, retrieval, fused, ranks = fold_questions, retrieval_train, fused_train, train_ranks
    for q in questions:
        qid = q["qid"]
        candidates = fused[qid]["candidates"][:80]
        if len(candidates) != 80 or len(set(candidates)) != 80:
            raise RuntimeError(f"Invalid Phase 4 top 80: {qid}")
        for name in RERANKERS:
            ranking = ranks[name][qid]
            if len(ranking) != 80 or set(ranking) != set(candidates):
                raise RuntimeError(f"Phase 4 {name} ranking/top-80 mismatch: {qid}")
        for name in ("bm25", "accent_char", "vietlegal_harrier", "vietnamese_embedding",
                     "nemotron", "query_memory", "query_exact"):
            if name not in retrieval[qid]["channels"]:
                raise RuntimeError(f"Missing retrieval channel {name}: {qid}")

    baseline = baseline_predictions(fold_questions, fused_train, train_ranks, weights)
    if abs(score_predictions(baseline, fold_questions)["recall"] -
           report["phase2_baseline_fold0"]["recall"]) > 1e-10:
        raise RuntimeError("Phase 2 baseline cannot be reproduced from caches")
    x, y, metadata, groups = make_rows(fold_questions, fused_train, retrieval_train, train_ranks, True)
    selected = report["selected_cross_fitted"]
    if (selected["C"], selected["alpha"], selected["promotion_margin"]) != (0.1, 1.0, 0.0):
        raise RuntimeError("Saved Phase 4 selected a different configuration")
    if (len(fold_questions) != report["training_questions"] or len(y) != report["training_pairs"]
            or int(y.sum()) != report["positive_pairs"] or x.shape[1] != report["feature_count"]
            or score_candidates({q["qid"]: fused_train[q["qid"]]["candidates"][:80]
                                 for q in fold_questions}, fold_questions)["candidate_recall"]
            != report["candidate_recall_fold0"]):
        raise RuntimeError("Reconstructed Phase 4 training data differs from report")
    scores = oof_scores(x, y, groups, metadata)
    phase4_pred, _ = protected_predictions(scores, metadata, baseline, 0.0)
    consensus_pred, changes = consensus_predictions(phase4_pred, scores, metadata, train_ranks)
    phase4_oof = {"metrics": score_predictions(phase4_pred, fold_questions),
                  "per_inner": per_inner(phase4_pred, fold_questions)}
    consensus_oof = {"metrics": score_predictions(consensus_pred, fold_questions),
                     "per_inner": per_inner(consensus_pred, fold_questions), **changes}
    if abs(phase4_oof["metrics"]["recall"] - selected["metrics"]["recall"]) > 1e-10 or any(
        abs(phase4_oof["per_inner"][str(i)]["recall"] - selected["per_inner"][str(i)]["recall"]) > 1e-10
        for i in range(5)
    ):
        raise RuntimeError("Phase 4 OOF cannot be reproduced; refusing to evaluate Phase 4.1")

    diagnostic = diagnose_oof(fold_questions, phase4_pred, consensus_pred, fused_train)
    if diagnostic["changed_answer_sets"] != changes["changed_answer_sets"]:
        raise RuntimeError("Diagnostic swap count differs from fixed consensus rule")
    delta = consensus_oof["metrics"]["recall"] - phase4_oof["metrics"]["recall"]
    if abs(diagnostic["mean_recall_delta"] - delta) > 1e-10:
        raise RuntimeError("Per-question recall deltas do not reproduce aggregate metric")
    for fold in range(5):
        key = str(fold)
        count = sum(inner_fold(q["question"]) == fold for q in fold_questions)
        actual = consensus_oof["per_inner"][key]["recall"] - phase4_oof["per_inner"][key]["recall"]
        if abs(diagnostic["per_inner"][key]["recall_delta"] / count - actual) > 1e-10:
            raise RuntimeError(f"Per-question recall deltas differ in inner fold {fold}")
    output = {"experiment_id": "phase41-cache-only-consensus-diagnostic",
              "phase4_report_sha256": hashlib.sha256((phase4_dir / "phase4_report.json").read_bytes()).hexdigest(),
              "test_sha256": report["test_sha256"], "training_questions": len(fold_questions),
              "rule": {"legal_top": LEGAL_TOP, "other_top": OTHER_TOP,
                       "weak_legal_after": WEAK_LEGAL_AFTER, "score_gap": SCORE_GAP,
                       "maximum_swaps_per_question": 1},
              "phase4_oof": phase4_oof, "consensus_oof": consensus_oof,
              "oof_delta": delta, "diagnostic": diagnostic,
              "phase4_to_top5_oracle_headroom": diagnostic["top5_oracle_recall_upper_bound"] - phase4_oof["metrics"]["recall"],
              "approved": False, "submission": None,
              "reason": "Diagnostic only; retain Phase 4 and do not submit this experiment",
              "warning": "OOF analysis after multiple experiments is not an unbiased private estimate."}
    write_json(work / REPORT_NAME, output)
    print(json.dumps({key: value for key, value in output.items() if key != "diagnostic"} |
                     {"diagnostic": {key: value for key, value in diagnostic.items() if key not in ("swaps", "missed_phase4_by_question")}},
                     ensure_ascii=False, indent=2))


In [ ]:
# Analysis only: no JSON/ZIP submission can be generated by this notebook.
main(PHASE4_OUTPUT, PHASE2_ARTIFACTS, WORK, PRIVATE_FILE)
result = read(WORK / 'phase41_consensus_diagnostic_report.json')
print('DIAGNOSTIC ONLY ? retain Phase 4; do not spend the last submission attempt.')
print('OOF Recall delta:', result['oof_delta'])
print('Swap outcomes:', result['diagnostic']['outcomes'])
print('Missed answers:', result['diagnostic']['missed_phase4'])
print('Top-5 oracle headroom:', result['phase4_to_top5_oracle_headroom'])
